## 6.7 Gradient Boosting and XGBoost

Gradient boosting is a different way of combining multiple decision tree models into one ensemble. 

In boosting, we start with a data set; then we train the first model; the first model makes predictions, then we look at the predictions, and what are the errors that this model is making. Based on these errors we train another model. Then this second model makes predictions again. This model also makes some errors. Then we train a third model that corrects the predictions of the second model; we repeat this, and do this for many itterations. Finally we combine multiple predictions into the **final prediction**. The training of the series of models is sequential. 

In this sequence of models, if we replace each model with a tree, we get the idea of boosted trees.

In [1]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 46.4 MB/s  0:00:01s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 18.7 MB/s  0:00:14 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /usr/local/python/3.14.2/bin/python3 -m pip install --upgrade pip


In [2]:
import xgboost as xgb

In [3]:
# Environment setup

import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

# Data source

# data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv'
data = "CreditScoring.csv"

In [4]:
# Load Data into environment

!wget $data

--2026-11-01 12:53:37--  http://creditscoring.csv/
Resolving creditscoring.csv (creditscoring.csv)... failed: Name or service not known.
wget: unable to resolve host address ‘creditscoring.csv’


In [5]:
df = pd.read_csv(data)
df.head()

,Status,Seniority,Home,Time,Age,Marital,Records,Job,Expenses,Income,Assets,Debt,Amount,Price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [6]:
# Prepare the data for ML processing

df.columns = df.columns.str.lower()

In [7]:
# Re-encoding the categorical variables

status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

df.status = df.status.map(status_values)

In [8]:
home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

In [9]:
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,ok,9,rent,60,30,married,no,freelance,73,129,0,0,800,846
1,ok,17,rent,60,58,widow,no,fixed,48,131,0,0,1000,1658
2,default,10,owner,36,46,married,yes,freelance,90,200,3000,0,2000,2985
3,ok,0,rent,60,24,single,no,fixed,63,182,2500,0,900,1325
4,ok,0,rent,36,26,single,no,fixed,46,107,0,0,310,910


In [10]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,763317.0,1060341.0,404382.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,8703625.0,10217569.0,6344253.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3500.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,166.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,99999999.0,99999999.0,99999999.0,5000.0,11140.0


In [11]:
for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)

In [12]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4421.0,4408.0,4437.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,131.0,5403.0,343.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,86.0,11573.0,1246.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3000.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,165.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,959.0,300000.0,30000.0,5000.0,11140.0


In [13]:
# Make a new dataframe from the old dataframe using every record, so long as the status is not 'unk'. 
# Then reset the dataframe index

df = df[df.status != 'unk'].reset_index(drop=True)

In [14]:
df.count()

status       4454
seniority    4454
home         4454
time         4454
age          4454
marital      4454
records      4454
job          4454
expenses     4454
income       4420
assets       4407
debt         4436
amount       4454
price        4454
dtype: int64

In [15]:
# Split source data in to Train / Validation / Test

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

In [16]:
# We reset the indexes so they are fresh and unique in each dataframe:

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [17]:
# Preparing the Target Variable

y_train = (df_train.status == 'default').astype('int').values
y_val = (df_val.status == 'default').astype('int').values
y_test = (df_test.status == 'default').astype('int').values

In [18]:
# Delete status from the dataframes

del df_train['status']
del df_val['status']
del df_test['status']

Explain here please

`DictVectorizer` one-hot encodes the categorical features - it creates columns like job=partime and records=no - and leaves the numeric ones unchanged.

In [38]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score

In [24]:
# fill the missing values with 0

train_dicts = df_train.fillna(0).to_dict(orient='records')

# Use DictVectorizer to convert the dictionaries into a feature matrix

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

You really need to know what this does

In [28]:
val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dicts)

### DMatrix

Wrap our training data in a special data structure internal to XGBoost called DMatrix
DMatrix() is optimised for training XGBoost models 
`X_train` is feature matrix
`y_train` is target variable

In [29]:
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [34]:
# intel xeon cpu 5-1650 v3 3.50GHz number of cores: ; number of threads: 12
# https://xgboost.readthedocs.io/en/latest/parameter.html

xgb_params = {
    'eta': 0.3, # learning rate; how fast our model learns
    'max_depth': 6, # controls the size of the trees; default value is 6
    'min_child_weight': 1, # the same as min_samples_weight; how many observations we should have in a leaf node; default is 1

    'objective': 'binary:logistic', # there are many types of objectives; xgb boost can be used for different things
    'nthread': 12, # the parameter for parallelising and controlling the number of threads

    'seed': 1,
    'verbosity': 1, # controls what kind of warnings we have when we train our ; 1 means show only warnings
}

model = xgb.train(xgb_params, dtrain, num_boost_round=200)

In [35]:
model.predict(dval)

array([4.31361720e-02, 4.27737366e-03, 6.75859279e-04, 9.65001211e-02,
       1.06018677e-03, 3.52140144e-03, 3.02975444e-04, 8.86651695e-01,
       7.88923874e-02, 1.87714177e-04, 1.40199602e-01, 9.93337035e-01,
       7.39668071e-01, 6.21446699e-04, 4.79650265e-03, 9.66348648e-01,
       6.52493036e-04, 7.03575373e-01, 2.45591067e-03, 4.67021428e-02,
       1.22672541e-03, 9.50312540e-02, 7.91552651e-04, 8.92462671e-01,
       9.53605831e-01, 5.45811979e-03, 7.28057563e-01, 9.51101445e-03,
       9.75149453e-01, 6.59366022e-04, 9.97828901e-01, 4.40825850e-01,
       8.96221623e-02, 7.87080005e-02, 4.88651101e-04, 1.47181518e-05,
       9.99814332e-01, 3.69971335e-01, 9.41287577e-01, 9.53366637e-01,
       3.15156989e-02, 9.95855629e-01, 3.51366447e-03, 3.14866394e-01,
       3.36176803e-04, 5.11976716e-04, 4.90001202e-01, 3.51968018e-04,
       4.59142635e-03, 6.04099594e-03, 6.19758648e-05, 1.90485697e-02,
       5.29917481e-04, 2.62436122e-01, 8.12831283e-01, 7.72655010e-02,
      

In [36]:
y_pred = model.predict(dval)

In [39]:
roc_auc_score(y_val, y_pred)

0.8045064299726334

Alexey got 0.8003166549611296

In [41]:
model = xgb.train(xgb_params, dtrain, num_boost_round=10)
y_pred = model.predict(dval)
roc_auc_score(y_val, y_pred)

0.8156256811411687

XGBoost is prone to overfitting, so we need to be careful with the size of the trees and the number that we train.